# Torghut system flow

**TL;DR.** This notebook checks whether typed equities, options, and Hyperliquid features are arriving and
whether their observable fields are usable. Equities use the latest completed/current market session;
options use 60 minutes; Hyperliquid uses 30 minutes. A closed exchange session is `expected_idle`, not a
false outage. Hyperliquid `event_ts` is the interval end; ingestion freshness comes from `ingest_ts` and
`source_lag_seconds`.

The notebook never scans raw `hyperliquid_bbo` and never substitutes fixtures in live mode.

## Context and method

All source queries are bounded, typed, parameterized, and implemented in `app.notebook_data` so notebook cells contain no hidden query logic.

In [ ]:
import json
import math

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display
from plotly.subplots import make_subplots

from app.notebook_data import (
    Window,
    adapter_from_environment,
    capital_authority,
    execution_evidence,
    flow_snapshot,
    mode_banner,
    strategy_lifecycle,
)

adapter = globals().get('adapter') or adapter_from_environment()
banner = mode_banner(adapter)
banner_color = '#9a6700' if adapter.mode == 'fixture' else '#b42318'
display(HTML(f'''<div style="padding:14px 18px;border:2px solid {banner_color};border-radius:8px;
  font-weight:700;background:#fff8e6;margin-bottom:14px">{banner}</div>'''))

def bounded_points(frame, series_column, *, time_column='event_ts', preferred=5000):
    # Deterministically thin display points while preserving each series.
    if frame.empty or len(frame) <= preferred:
        return frame.copy()
    series_count = max(1, frame[series_column].nunique())
    per_series = max(2, preferred // series_count)
    chunks = []
    for _, group in frame.sort_values(time_column).groupby(series_column, sort=True):
        stride = max(1, math.ceil(len(group) / per_series))
        chunks.append(group.iloc[::stride])
    result = pd.concat(chunks, ignore_index=True)
    if len(result) > 10000:
        raise ValueError('figure point cap exceeded after deterministic thinning')
    return result

def bounded_rows(frame, *, preferred=5000):
    if frame.empty or len(frame) <= preferred:
        return frame.copy()
    stride = max(1, math.ceil(len(frame) / preferred))
    result = frame.iloc[::stride].head(10000).copy()
    if len(result) > 10000:
        raise ValueError('figure point cap exceeded after deterministic thinning')
    return result

def snapshot_frame(snapshot, name):
    if name not in snapshot.datasets:
        return pd.DataFrame()
    return snapshot.frame(name)

def empty_panel(message):
    display(HTML(f'<div style="padding:18px;border:1px solid #d0d5dd;border-radius:8px;color:#475467">{message}</div>'))

In [ ]:
snapshots = {
    'equities': flow_snapshot('equities', Window.sessions(), adapter=adapter),
    'options': flow_snapshot('options', Window.minutes(60), adapter=adapter),
    'hyperliquid': flow_snapshot('hyperliquid', Window.minutes(30), adapter=adapter),
}
summary_rows = []
for lane, snapshot in snapshots.items():
    row = snapshot.provenance().iloc[0].to_dict()
    age = snapshot.captured_at - snapshot.source_watermark if snapshot.source_watermark else None
    row.update({
        'lane': lane,
        'persisted_rows': len(snapshot.datasets.get(lane, ())),
        'wall_clock_age_seconds': age.total_seconds() if age else None,
    })
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows)
display(summary[['lane', 'quality', 'captured_at_utc', 'source_watermark_utc',
                 'wall_clock_age_seconds', 'persisted_rows', 'truncated', 'query_identifier']])

## Equities — EMA, RSI, and wall-clock age

In [ ]:
equities = snapshot_frame(snapshots['equities'], 'equities')
if equities.empty:
    empty_panel('No equities rows: ' + '; '.join(snapshots['equities'].messages))
else:
    equities['event_ts'] = pd.to_datetime(equities['event_ts'], utc=True)
    equities_plot = bounded_points(equities, 'symbol')
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                        subplot_titles=('EMA 12 vs EMA 26', 'RSI 14'))
    for symbol, group in equities_plot.groupby('symbol'):
        fig.add_trace(go.Scatter(x=group.event_ts, y=group.ema12, name=f'{symbol} EMA12'), row=1, col=1)
        fig.add_trace(go.Scatter(x=group.event_ts, y=group.ema26, name=f'{symbol} EMA26', line={'dash': 'dot'}), row=1, col=1)
        fig.add_trace(go.Scatter(x=group.event_ts, y=group.rsi14, name=f'{symbol} RSI14'), row=2, col=1)
    fig.add_hline(y=70, line_dash='dash', line_color='#b42318', row=2, col=1)
    fig.add_hline(y=30, line_dash='dash', line_color='#027a48', row=2, col=1)
    fig.update_layout(title='Latest/current equities session — typed minute aggregates', height=760, hovermode='x unified')
    fig.update_yaxes(title_text='price', row=1, col=1)
    fig.update_yaxes(title_text='RSI', range=[0, 100], row=2, col=1)
    fig.show()
    watermark = snapshots['equities'].source_watermark
    age = snapshots['equities'].captured_at - watermark if watermark else None
    display(pd.DataFrame([{'market_session_state': snapshots['equities'].quality,
                           'wall_clock_age_seconds': age.total_seconds() if age else None,
                           'latest_event_ts': equities.event_ts.max()}]))

## Options — ATM IV, skew, and snapshot coverage

In [ ]:
options = snapshot_frame(snapshots['options'], 'options')
if options.empty:
    empty_panel('No options rows: ' + '; '.join(snapshots['options'].messages))
else:
    options['event_ts'] = pd.to_datetime(options['event_ts'], utc=True)
    options_plot = bounded_points(options, 'underlying_symbol')
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                        subplot_titles=('ATM implied volatility', '25-delta call/put skew', 'Snapshot coverage'))
    for symbol, group in options_plot.groupby('underlying_symbol'):
        fig.add_trace(go.Scatter(x=group.event_ts, y=group.atm_iv, name=f'{symbol} ATM IV'), row=1, col=1)
        fig.add_trace(go.Scatter(x=group.event_ts, y=group.call_put_skew_25d, name=f'{symbol} skew'), row=2, col=1)
        fig.add_trace(go.Scatter(x=group.event_ts, y=group.snapshot_coverage_ratio, name=f'{symbol} snapshot coverage'), row=3, col=1)
        fig.add_trace(go.Scatter(x=group.event_ts, y=group.hot_contract_coverage_ratio, name=f'{symbol} hot-contract coverage', line={'dash': 'dot'}), row=3, col=1)
    fig.update_layout(title='Options surface — last 60 minutes', height=820, hovermode='x unified')
    fig.update_yaxes(tickformat='.1%', row=1, col=1)
    fig.update_yaxes(tickformat='.1%', row=3, col=1, range=[0, 1.05])
    fig.show()
    display(options.groupby('feature_quality_status', dropna=False).size().rename('rows').reset_index())

## Hyperliquid — latest-ingest interval state

In [ ]:
hyper = snapshot_frame(snapshots['hyperliquid'], 'hyperliquid')
if hyper.empty:
    empty_panel('No Hyperliquid rows: ' + '; '.join(snapshots['hyperliquid'].messages))
else:
    hyper['event_ts'] = pd.to_datetime(hyper['event_ts'], utc=True)
    hyper['series'] = hyper['coin'].astype(str) + ' ' + hyper['interval'].astype(str)
    hyper_plot = bounded_points(hyper, 'series')
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                        subplot_titles=('Price by deduplicated interval', 'Regime', 'Source lag (seconds)'))
    for series, group in hyper_plot.groupby('series'):
        fig.add_trace(go.Scatter(x=group.event_ts, y=group.price, name=series), row=1, col=1)
        fig.add_trace(go.Scatter(x=group.event_ts, y=group.regime, name=f'{series} regime', mode='markers'), row=2, col=1)
        fig.add_trace(go.Scatter(x=group.event_ts, y=group.source_lag_seconds, name=f'{series} lag'), row=3, col=1)
    fig.update_layout(title='Hyperliquid typed features — event_ts is interval end', height=900, hovermode='x unified')
    fig.update_yaxes(title_text='price', row=1, col=1)
    fig.update_yaxes(title_text='regime', row=2, col=1)
    fig.update_yaxes(title_text='seconds', row=3, col=1)
    fig.show()
    display(hyper.groupby(['coin', 'interval', 'regime'], dropna=False).size().rename('rows').reset_index())

## Takeaways

Use the quality state, source watermark, wall-clock age, and explicit empty-state message together. A rendered chart proves observability, not profitability or capital authority.